# Mini caso 5 — Generación automática de fichas documentales

Este notebook plantea el mini caso de generación automática de fichas/resúmenes documentales para el corpus RTVE 23-F.

El objetivo no es generar todavía fichas definitivas para todos los documentos, sino validar si el caso tiene sentido y dejar planteada la metodología para la entrega final.

La idea principal es transformar cada documento en una ficha estructurada que facilite su lectura, comparación y reutilización en otros mini casos.

## 1. Carga de datos

Se utiliza como base el archivo limpio generado en la fase de limpieza general:

`data/processed/rtve_corpus_clean_base.csv`

Este archivo contiene el texto limpio (`text_clean_base`), el resumen original (`summary`), metadatos básicos y enlaces de trazabilidad.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rtve_corpus_clean_base.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape}")

df.head()

Dataset cargado: data/processed/rtve_corpus_clean_base.csv
Shape: (167, 25)


,doc_id,source_document_id,title,pages,detail_url,pdf_url,summary,keywords,text_full,text_clean_base,text_length_chars,text_length_words,text_clean_length_chars,text_clean_length_words,moncloa_id,moncloa_section,moncloa_subsection,final_match_status,coverage_moncloa,alpha_ratio,digit_ratio,uppercase_ratio,weird_char_ratio,n_title_years,title_main_year
0,rtve_1860,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrer...,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones pa...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n-...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n-...,3934,640,3934,640,moncloa_0099,defensa,cni,high_confidence_match,True,0.777834,0.013726,0.147386,0.000000,1,1982.0
1,rtve_1859,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrer...,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el C...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SE...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SE...,6417,1018,6417,1018,moncloa_0098,defensa,cni,high_confidence_match,True,0.781985,0.009506,0.195895,0.000156,1,1982.0
2,rtve_1858,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrer...,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Ju...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,8183,1347,8183,1347,moncloa_0097,defensa,cni,high_confidence_match,True,0.784920,0.011487,0.124085,0.000611,1,1982.0
3,rtve_1857,1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,6,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_25_de_febrer...,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa al juicio por los suc...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,11151,1826,11151,1826,moncloa_0096,defensa,cni,high_confidence_match,True,0.789257,0.008250,0.128167,0.000538,1,1982.0
4,rtve_1856,1

## 2. Validación del resumen disponible

El corpus ya contiene una columna `summary`, pero el EDA general indicó que estos resúmenes parecen recortados.

Antes de plantear una generación automática de fichas, se revisa si el resumen disponible es suficiente o si realmente existe una oportunidad de mejora.

In [2]:
summary_review = {
    "n_documents": len(df),
    "n_summary_available": df["summary"].notna().sum(),
    "n_empty_summary": (df["summary"].fillna("").str.strip() == "").sum(),
    "min_summary_length_chars": int(df["summary"].fillna("").str.len().min()),
    "median_summary_length_chars": float(df["summary"].fillna("").str.len().median()),
    "mean_summary_length_chars": float(df["summary"].fillna("").str.len().mean()),
    "max_summary_length_chars": int(df["summary"].fillna("").str.len().max()),
    "n_summary_ends_with_ellipsis": int(df["summary"].fillna("").str.endswith("...").sum()),
}

pd.DataFrame(summary_review.items(), columns=["check", "value"])

,check,value
0,n_documents,167.0
1,n_summary_available,167.0
2,n_empty_summary,0.0
3,min_summary_length_chars,303.0
4,median_summary_length_chars,303.0
5,mean_summary_length_chars,303.0
6,max_summary_length_chars,303.0
7,n_summary_ends_with_ellipsis,167.0


In [3]:
df[["doc_id", "title", "summary"]].head(5)

,doc_id,title,summary
0,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones pa..."
1,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el C...
2,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Ju...
3,rtve_1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa al juicio por los suc...
4,rtve_1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la sesión del 26 de febrer...


**Comprobación adicional**

In [8]:
# Comprobación explícita: los puntos suspensivos forman parte real del texto,
# no son solo una truncación visual de Jupyter.

summary_check = df[["doc_id", "summary"]].copy()
summary_check["summary_repr"] = summary_check["summary"].apply(repr)
summary_check["summary_last_20_chars"] = summary_check["summary"].str[-20:]
summary_check["ends_with_ellipsis"] = summary_check["summary"].str.endswith("...")

summary_check.head(10)

,doc_id,summary,summary_repr,summary_last_20_chars,ends_with_ellipsis
0,rtve_1860,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones pa...","'El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones p...",pecialmente en re...,True
1,rtve_1859,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el C...,'Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el ...,largo de las ses...,True
2,rtve_1858,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Ju...,'Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de J...,y fuerzas militar...,True
3,rtve_1857,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa al juicio por los suc...,'El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa al juicio por los su...,"s y acusados, mar...",True
4,rtve_1856,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la sesión del 26 de febrer...,'Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la sesión del 26 de febre...,de implicación de...,True
5,rtve_1855,El documento aborda el desarrollo y contexto de la causa judicial 2/81 relacionada con un intento de golpe de estado. En la reanudación ...,'El documento aborda el desarrollo y contexto de la causa judicial 2/81 relacionada con un intento de golpe de estado. En la reanudación...,encionar otros po...,True
6,rtve_1854,"El informe C/DI/4339/16-03-82 analiza la situación jurídica y social derivada de la causa 2/81. En la vista pública, aunque se observan ...","'El informe C/DI/4339/16-03-82 analiza la situación jurídica y social derivada de la causa 2/81. En la vista pública, aunque se observan...","res, generando pr...",True
7,rtve_1853,Resumen global del documento sobre el intento de golpe de Estado del 23-F:\n\nEl documento recoge en detalle las sesiones del juicio mil...,'Resumen global del documento sobre el intento de golpe de Estado del 23-F:\n\nEl documento recoge en detalle las sesiones del juicio mi...,on el proceso. Du...,True
8,rtve_1852,Resumen global:\n\nLas sesiones judiciales de marzo de 1982 se centraron en la defensa y esclarecimiento de los hechos relacionados con ...,'Resumen global:\n\nLas sesiones judiciales de marzo de 1982 se centraron en la defensa y esclarecimiento de los hechos relacionados con...,presentaron numer...,True
9,rtve_1851,"Resumen global:\n\nEl documento corresponde al desarrollo del juicio en la causa 2/81 del Consejo Supremo de Justicia Militar, centrado ...","'Resumen global:\n\nEl documento corresponde al desarrollo del juicio en la causa 2/81 del Consejo Supremo de Justicia Militar, centrado...",stran las defensa...,True


**Lectura del bloque**

El campo `summary` está disponible para los 167 documentos y no contiene valores vacíos. Sin embargo, todos los resúmenes tienen exactamente 303 caracteres y terminan en `...`.

La comprobación adicional confirma que los puntos suspensivos forman parte real del campo y no son solo una truncación visual de Jupyter. Por tanto, el `summary` no debe interpretarse como un resumen completo del documento, sino como una vista preliminar recortada de forma sistemática.

Esta limitación justifica el mini caso: generar fichas documentales más completas a partir de `text_clean_base`, especialmente para documentos largos o complejos, cuyos resúmenes actuales no representan adecuadamente el contenido completo.

## 3. Comparación entre resumen disponible y texto completo

Una ficha documental solo tiene sentido si el texto completo aporta mucha más información que el resumen existente.

Por ello se compara la longitud del resumen con la longitud del texto limpio.

In [4]:
df_summary_vs_text = df.copy()

df_summary_vs_text["summary_length_words"] = (
    df_summary_vs_text["summary"]
    .fillna("")
    .str.split()
    .str.len()
)

df_summary_vs_text["summary_to_text_ratio"] = (
    df_summary_vs_text["summary_length_words"] / df_summary_vs_text["text_clean_length_words"]
).replace([np.inf, -np.inf], np.nan)

comparison_stats = {
    "median_summary_words": float(df_summary_vs_text["summary_length_words"].median()),
    "mean_summary_words": float(df_summary_vs_text["summary_length_words"].mean()),
    "median_text_clean_words": float(df_summary_vs_text["text_clean_length_words"].median()),
    "mean_text_clean_words": float(df_summary_vs_text["text_clean_length_words"].mean()),
    "median_summary_to_text_ratio": float(df_summary_vs_text["summary_to_text_ratio"].median()),
    "mean_summary_to_text_ratio": float(df_summary_vs_text["summary_to_text_ratio"].mean()),
}

pd.DataFrame(comparison_stats.items(), columns=["metric", "value"])

,metric,value
0,median_summary_words,49.000000
1,mean_summary_words,48.568862
2,median_text_clean_words,579.000000
3,mean_text_clean_words,2075.443114
4,median_summary_to_text_ratio,0.081911
5,mean_summary_to_text_ratio,0.143744


**Lectura del bloque**

La comparación entre el resumen disponible y el texto limpio confirma que el `summary` actual representa solo una pequeña parte del contenido.

La mediana del resumen es de 49 palabras, mientras que la mediana del texto limpio es de 579 palabras. En promedio, los textos completos son mucho más extensos: 2.075 palabras frente a unas 49 palabras de resumen.

Además, la mediana del ratio `summary_to_text_ratio` es aproximadamente 0,082. Es decir, el resumen representa alrededor del 8 % del texto limpio en el documento mediano. En documentos largos, este porcentaje cae por debajo del 1 %.

Por tanto, una ficha documental generada a partir del texto completo aportaría una representación más útil que el resumen actual.

In [5]:
df_summary_vs_text[
    [
        "doc_id",
        "title",
        "summary_length_words",
        "text_clean_length_words",
        "summary_to_text_ratio",
        "pdf_url",
    ]
].sort_values("text_clean_length_words", ascending=False).head(10)

,doc_id,title,summary_length_words,text_clean_length_words,summary_to_text_ratio,pdf_url
161,rtve_1699,Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.,44,95293,0.000462,https://www.rtve.es/contenidos/documentos/23f-desclasificado/07_2026_transcripcion_de_cintas_grabadas_con_conversaciones_telefonicas_con...
159,rtve_1701,Télex interiores y de agencias recibidos en 2ª sección EM el día 23-F informando del estado de situación (23 de febrero de 1981).,46,19857,0.002317,https://www.rtve.es/contenidos/documentos/23f-desclasificado/09_1981_telex_interiores_y_de_agencias_recibidos_en_2%C2%AA_seccion_em_el_d...
63,rtve_1797,Investigación y declaraciones personal AOME por JDDI (9 de abril de 1981).,51,14485,0.003521,https://www.rtve.es/contenidos/documentos/23f-desclasificado/36_1981_investigacion_y_declaraciones_personal_aome_por_jddi_9_de_abril_de_...
76,rtve_1784,Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).,51,13639,0.003739,https://www.rtve.es/contenidos/documentos/23f-desclasificado/23_1981_policia_nacional_informe_de_situacion_marca_reservado_confidencial_...
74,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",48,11080,0.004332,https://www.rtve.es/contenidos/documentos/23f-desclasificado/25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesal...
126,rtve_1734,"""Nota Informativa sobre la repercusión en prensa del arresto de Tejero en 1978 cuando era Jefe de la Comandancia de Málaga",48,10253,0.004682,https://www.rtve.es/contenidos/documentos/23f-desclasificado/12_1978_nota_informativa_sobre_la_repercusion_en_prensa_del_arresto_de_teje...
50,rtve_1810,Relato de los sucesos de los días 23 y 24 de febrero.,46,8965,0.005131,https://www.rtve.es/contenidos/documentos/23f-desclasificado/49_2026_relato_de_los_sucesos_de_los_dias_23_y_24_de_febrero.pdf
157,rtve_1703,Semestral de la amenaza interior (10 de febrero de 1981; 9 de marzo de 1981).,44,8714,0.005049,https://www.rtve.es/contenidos/documentos/23f-desclasificado/101_1981_semestral_de_la_amenaza_interior_10_de_febrero_de_1981_9_de_marzo_...
93,rtve_1767,"""Informe de las distintas Jefaturas Superiores: Comisaría General de Información. """"Situación actual en las distintas regiones policiale...",50,7189,0.006955,https://www.rtve.es/contenidos/documentos/23f-desclasificado/15_1981_informe_de_las_distintas_jefaturas_superiores_comisaria_general_de_...
163,rtve_1697,"""Documentación con una presunta planificación del golpe",47,4882,0.009627,https://www.rtve.es/contenidos/documentos/23f-desclasificado/05_1980_documentacion_con_una_presunta_planificacion_del_golpe_manuscrita_1...


## 4. Propuesta de ficha documental

La ficha documental se plantea como una salida estructurada por documento.

Campos propuestos:

- `doc_id`: identificador trazable.
- `title`: título del documento.
- `resumen_corto`: síntesis breve del contenido.
- `resumen_extendido`: síntesis más completa.
- `actores_principales`: personas mencionadas o relevantes.
- `instituciones_mencionadas`: instituciones u organismos.
- `fechas_relevantes`: fechas detectadas o mencionadas.
- `temas_clave`: temas principales del documento.
- `moncloa_section`: sección institucional, si existe.
- `pdf_url`: enlace al documento original.

Esta salida podría alimentar otros mini casos: interpretación de clusters, enriquecimiento del grafo de entidades, resultados del asistente semántico o revisión de errores de clasificación.

In [6]:
ficha_schema = pd.DataFrame({
    "field": [
        "doc_id",
        "title",
        "resumen_corto",
        "resumen_extendido",
        "actores_principales",
        "instituciones_mencionadas",
        "fechas_relevantes",
        "temas_clave",
        "moncloa_section",
        "pdf_url",
    ],
    "description": [
        "Identificador único del documento.",
        "Título original del documento.",
        "Resumen breve generado a partir del texto limpio.",
        "Resumen más detallado para documentos complejos o largos.",
        "Personas o actores destacados.",
        "Instituciones, organismos o unidades mencionadas.",
        "Fechas relevantes detectadas en el texto.",
        "Etiquetas temáticas interpretables.",
        "Sección institucional disponible desde la comparación con Moncloa.",
        "Enlace al PDF original.",
    ]
})

ficha_schema

,field,description
0,doc_id,Identificador único del documento.
1,title,Título original del documento.
2,resumen_corto,Resumen breve generado a partir del texto limpio.
3,resumen_extendido,Resumen más detallado para documentos complejos o largos.
4,actores_principales,Personas o actores destacados.
5,instituciones_mencionadas,"Instituciones, organismos o unidades mencionadas."
6,fechas_relevantes,Fechas relevantes detectadas en el texto.
7,temas_clave,Etiquetas temáticas interpretables.
8,moncloa_section,Sección institucional disponible desde la comparación con Moncloa.
9,pdf_url,Enlace al PDF original.


## 5. Metodología futura

Para la entrega final se plantea el siguiente flujo:

1. Seleccionar `text_clean_base` como fuente principal.
2. Dividir documentos largos en fragmentos si superan un umbral de longitud.
3. Generar una ficha estructurada por documento.
4. Extraer o generar:
   - resumen corto;
   - resumen extendido;
   - actores o entidades principales;
   - instituciones;
   - fechas relevantes;
   - temas clave.
5. Mantener siempre trazabilidad con `doc_id`, `title` y `pdf_url`.
6. Validar manualmente una muestra de fichas para comprobar coherencia y ausencia de información inventada.

La generación podría abordarse con dos niveles:

- enfoque extractivo, seleccionando frases relevantes del propio texto;
- enfoque generativo asistido por LLM, siempre condicionado a no introducir información externa al documento.

In [7]:
planned_methodology = {
    "input": "text_clean_base",
    "unit": "documento completo o fragmentos para textos largos",
    "outputs": [
        "resumen_corto",
        "resumen_extendido",
        "actores_principales",
        "instituciones_mencionadas",
        "fechas_relevantes",
        "temas_clave",
    ],
    "baseline": "resumen extractivo basado en frases relevantes",
    "advanced_option": "LLM con prompt controlado y trazabilidad al documento",
    "validation": [
        "revisión manual de muestra",
        "comprobación de que la ficha no introduce información externa",
        "comparación con summary original como baseline limitado",
    ],
    "traceability": [
        "doc_id",
        "title",
        "pdf_url",
    ],
}

planned_methodology

{'input': 'text_clean_base',
 'unit': 'documento completo o fragmentos para textos largos',
 'outputs': ['resumen_corto',
  'resumen_extendido',
  'actores_principales',
  'instituciones_mencionadas',
  'fechas_relevantes',
  'temas_clave'],
 'baseline': 'resumen extractivo basado en frases relevantes',
 'advanced_option': 'LLM con prompt controlado y trazabilidad al documento',
 'validation': ['revisión manual de muestra',
  'comprobación de que la ficha no introduce información externa',
  'comparación con summary original como baseline limitado'],
 'traceability': ['doc_id', 'title', 'pdf_url']}

## 6. Conexión con otros mini casos

Este mini caso puede funcionar como una capa transversal de enriquecimiento documental.

Las fichas generadas podrían utilizarse para:

- interpretar mejor los clusters del Mini caso 2;
- alimentar o validar entidades del grafo del Mini caso 3;
- mostrar una vista rápida de los documentos recuperados por el asistente semántico del Mini caso 4;
- ayudar a revisar errores o predicciones dudosas del clasificador institucional del Mini caso 1.

Por tanto, aunque se presenta como mini caso independiente, su resultado puede servir como apoyo para el resto del proyecto.

## 7. Riesgos y limitaciones

Los principales riesgos del mini caso son:

- generación de información no presente en el documento si se usa un LLM sin control;
- pérdida de matices en documentos muy largos;
- ruido OCR que afecte a la calidad del resumen;
- dificultad para evaluar automáticamente la calidad de las fichas;
- posible dependencia de revisión manual para validar resultados.

Para reducir estos riesgos, la metodología debe priorizar la trazabilidad, el uso del texto original como única fuente y la validación manual de una muestra de documentos.

## 8. Conclusión de viabilidad

El mini caso es viable porque los 167 documentos del corpus tienen texto limpio disponible y, al mismo tiempo, el resumen actualmente incluido en el dataset presenta una limitación clara.

Aunque la columna `summary` está completa para todos los documentos, la validación muestra que todos los resúmenes tienen exactamente 303 caracteres y terminan en `...`. Esto confirma que se trata de resúmenes recortados de forma sistemática, no de resúmenes completos del contenido documental.

La comparación entre `summary` y `text_clean_base` refuerza esta conclusión. La mediana del resumen es de 49 palabras, mientras que la mediana del texto limpio es de 579 palabras. Además, el ratio mediano entre resumen y texto completo es aproximadamente 0,082, por lo que el resumen representa solo una pequeña parte del contenido real del documento. En documentos especialmente largos, esta proporción cae por debajo del 1 %.

Por tanto, existe una oportunidad clara de mejora: generar fichas documentales estructuradas a partir del texto completo limpio. Estas fichas podrían incluir resumen corto, resumen extendido, actores principales, instituciones mencionadas, fechas relevantes, temas clave y enlace al PDF original.

Este mini caso no sustituye a los demás, sino que puede funcionar como una capa transversal de enriquecimiento documental. Las fichas generadas podrían ayudar a interpretar clusters, enriquecer grafos de entidades, revisar resultados de clasificación y mejorar la experiencia del asistente documental semántico.

En la fase final será necesario decidir si se utilizará un enfoque extractivo, un enfoque generativo controlado o una combinación de ambos. En cualquier caso, la metodología deberá mantener la trazabilidad al documento original y evitar introducir información que no esté respaldada por el propio texto.